# 02.5_v2 — Candidate Pool Review

Review the URL pool before running the expensive fetch + screenshot pipeline.
Check type coverage, source distribution, TLD spread.

In [ ]:
import json
from pathlib import Path
from collections import Counter
from urllib.parse import urlparse

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from io import BytesIO
from IPython.display import display, Image as IPImage

candidates_path = Path('../data/raw/wdc_candidate_urls.jsonl')
records = []
with open(candidates_path) as f:
    for line in f:
        records.append(json.loads(line))

types = Counter(r['schema_type'] for r in records)
sources = Counter(r.get('source', 'wdc') for r in records)
tlds = Counter()
for r in records:
    host = urlparse(r['url']).netloc
    if host.endswith('.co.uk'): tlds['.co.uk'] += 1
    elif host.endswith('.com.au'): tlds['.com.au'] += 1
    elif host.endswith('.co.nz'): tlds['.co.nz'] += 1
    else:
        parts = host.rsplit('.', 1)
        tlds[f'.{parts[-1]}'] += 1

print(f'Total: {len(records):,} URLs | {len(types)} types | {len(sources)} sources')

In [ ]:
# Type distribution vs targets
with open('../config/type_distribution.json') as f:
    cfg = json.load(f)
weights = {k: v['weight'] for k, v in cfg['types'].items()}
total_w = sum(weights.values())
targets = {t: int(20_000 * w / total_w) for t, w in weights.items()}
# Need 4x oversample to account for quality filter
need_4x = {t: v * 4 for t, v in targets.items()}

fig, ax = plt.subplots(figsize=(12, 6))
all_types = sorted(set(list(types.keys()) + list(need_4x.keys())),
                   key=lambda t: types.get(t, 0), reverse=True)
x = range(len(all_types))
have = [types.get(t, 0) for t in all_types]
need = [need_4x.get(t, 0) for t in all_types]
ax.bar(x, have, label='Have', alpha=0.7)
ax.bar(x, need, label='Need (4x oversample)', alpha=0.3, color='red')
ax.set_xticks(x)
ax.set_xticklabels(all_types, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('URL count')
ax.set_title('Candidate Pool: Have vs Need')
ax.legend()
plt.tight_layout()

buf = BytesIO()
fig.savefig(buf, format='png', dpi=100)
buf.seek(0)
display(IPImage(data=buf.getvalue()))
plt.close(fig)

In [ ]:
# Coverage table
print(f'{"Type":25s} {"Have":>8s} {"Need(4x)":>8s} {"Target":>8s} {"Status"}')
print('-' * 70)
for t in all_types:
    h = types.get(t, 0)
    n = need_4x.get(t, 0)
    tgt = targets.get(t, 0)
    status = 'OK' if h >= n else ('LOW' if h >= tgt else 'CRITICAL')
    print(f'{t:25s} {h:8,} {n:8,} {tgt:8,} {status}')

In [ ]:
# Source and TLD pies
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.pie(sources.values(), labels=sources.keys(), autopct='%1.0f%%')
ax1.set_title('By Source')
top_tlds = dict(tlds.most_common(8))
ax2.pie(top_tlds.values(), labels=top_tlds.keys(), autopct='%1.0f%%')
ax2.set_title('By TLD (top 8)')
plt.tight_layout()

buf = BytesIO()
fig.savefig(buf, format='png', dpi=100)
buf.seek(0)
display(IPImage(data=buf.getvalue()))
plt.close(fig)

In [ ]:
# Sample URLs per type for spot-checking
import random
by_type = {}
for r in records:
    by_type.setdefault(r['schema_type'], []).append(r)

for t in sorted(by_type):
    sample = random.sample(by_type[t], min(3, len(by_type[t])))
    print(f'\n{t} ({len(by_type[t]):,} URLs):')
    for r in sample:
        print(f'  {r["url"][:80]}')